# SAM-3D Body Pipeline

Runs pose estimation on an image and saves outputs to `data/output/`.

**First time?** Run cells 1–3 to install everything. After that you only need cells 4+.

## ① Setup

In [ ]:
# Can skip if not using your Google Drive 

# from google.colab import drive, files
# drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Clone SAM-3D Body and SAM3 (Meta's upstream repos)
!git clone https://github.com/facebookresearch/sam-3d-body.git
!git clone https://github.com/facebookresearch/sam3.git

fatal: destination path 'sam-3d-body' already exists and is not an empty directory.
fatal: destination path 'sam3' already exists and is not an empty directory.


In [5]:
# Install SAM3 in editable mode (must be done before other deps)
%cd sam3
!pip install -e . --quiet
%cd ..

/content/sam3
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 83.2 MB/s eta 0:00:00:00:0100:01
  Building editable for sam3 (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einsta

In [ ]:
# Install all project dependencies
!pip install -r sam3dbody_pipeline/requirements.txt --quiet

# SAM-3D Body specific deps
!pip install pytorch-lightning pyrender yacs scikit-image einops timm hydra-core roma --quiet
!pip install 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' --no-build-isolation --no-deps --quiet
!pip install git+https://github.com/microsoft/MoGe.git --quiet

# Pin numpy (required by SAM3D)
!pip install numpy==1.26.4 --force-reinstall --quiet

print('All dependencies installed. Restart kernel before continuing.')

ERROR: Could not open requirements file: [Errno 2] No such file or directory: '../requirements.txt'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 37.8 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.8/740.8 kB 35.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  G

## ② Load Model

In [17]:
!ls sam3dbody_pipeline  

data		     README.md	       sam-3d-body
joint_analysis.html  requirements.txt  SAM3DB_Official_README.md
notebooks	     sam3	       SAM3DB_Technical_Guide.md
Official_INSTALL.md  sam3db


In [ ]:
import sys
sys.path.insert(0, '..')

from sam3db.pipeline import load_model

# Reads HF_TOKEN from .env automatically
estimator, visualizer = load_model()

ModuleNotFoundError: No module named 'numpy.char'

## ③ Run Inference

In [ ]:
import cv2
import matplotlib.pyplot as plt
from sam3db.pipeline import run_inference

IMAGE_PATH = '../data/input/your_image.png'  # ← change this

img_cv2 = cv2.imread(IMAGE_PATH)
img_rgb = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)

outputs = run_inference(estimator, IMAGE_PATH)

plt.figure(figsize=(8, 5))
plt.imshow(img_rgb)
plt.axis('off')
plt.title('Input Image')
plt.show()

## ④ Save Outputs

In [ ]:
import pickle
import os
from pathlib import Path

image_name = Path(IMAGE_PATH).stem
output_path = f'../data/output/{image_name}_outputs.pkl'

os.makedirs('../data/output', exist_ok=True)
with open(output_path, 'wb') as f:
    pickle.dump(outputs, f)

print(f'✅ Saved to {output_path}')

## ⑤ 2D Keypoint Visualization

In [ ]:
# Uses the upstream utils (from sam-3d-body/notebook/)
import sys
sys.path.insert(0, '../sam-3d-body/notebook')
from utils import visualize_2d_results, display_results_grid

if outputs:
    vis_results = visualize_2d_results(img_cv2, outputs, visualizer)
    titles = [f'Person {i} — 2D Keypoints' for i in range(len(vis_results))]
    display_results_grid(vis_results, titles, figsize_per_image=(6, 6))
else:
    print('No people detected.')